# Quick run — three ROS 2 / Zenoh samples

This notebook builds (if needed) and runs each sample for a few seconds:

1. **Traditional DDS** — Cyclone DDS pub/sub (**Docker** or host ROS)
2. **rmw_zenoh** — Zenoh as RMW + `zenohd`
3. **DDS + Bridge** — local DDS talker → bridge → Rust subscriber

- **Sample 1 in Docker** — only needs Docker (no host ROS install)
- **Samples 2–3** — host ROS Jazzy, colcon, cargo, Docker

Run cells top-to-bottom. Use the cleanup cell when finished.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    REPO = NOTEBOOK_DIR.parent
else:
    REPO = NOTEBOOK_DIR

sys.path.insert(0, str(REPO / "notebooks"))

import demo_runner as dr
from demo_runner import (
    build_all,
    build_sample1_docker,
    check_prerequisites,
    print_results,
    run_all_demos,
    run_sample1_demo,
    run_sample1_docker_demo,
    run_sample2_demo,
    run_sample3_demo,
    zenohd_down,
    zenohd_up,
)

check_prerequisites()

## Sample 1 — Docker (Jazzy container)

Builds `ros:jazzy-ros-base` image with `demo_nodes`, runs talker + listener inside the container. **No host ROS required.**

In [ ]:
# Build sample 1 Docker image (first run may take a few minutes)
print(build_sample1_docker()[-1500:])

In [ ]:
# Run sample 1 inside the container (~6 s)
print(run_sample1_docker_demo(duration_sec=6, ros_domain_id=42))

## Build all samples

Skips quickly if already built. Re-run after changing C++ or Rust source.

In [ ]:
for line in build_all():
    if line.strip():
        print(line[-2000:])  # tail of colcon/cargo output
print("Build complete.")

## Run all three (sequential)

Each demo runs ~6–10 s, then processes are stopped automatically.

In [ ]:
results = run_all_demos(duration_sec=6, stop_router_after=True)
print_results(results)

## Or run individually

In [ ]:
# Sample 1 — Traditional DDS (host ROS; skip if using Docker above)
s1 = run_sample1_demo(duration_sec=6)
print(s1.tail_logs())
s1.stop_all()

In [ ]:
# Sample 2 — rmw_zenoh (starts zenohd)
s2 = run_sample2_demo(duration_sec=8)
print(s2.tail_logs())
s2.stop_all()

In [ ]:
# Sample 3 — DDS + Bridge + Rust sub (starts zenohd)
s3 = run_sample3_demo(duration_sec=10)
print(s3.tail_logs())
s3.stop_all()

## Cleanup

Stop `zenohd` and any stray demo processes.

In [ ]:
zenohd_down()
print("zenohd stopped.")